In [1]:
#Librerias
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
import time


## GPU Availability and Performance Validation in TensorFlow

Before deploying deep learning models, it is essential to verify that the computing environment is properly configured to leverage GPU acceleration.

In this validation, we perform two complementary steps:

1. **Device discovery**  
   We list all physical devices recognized by TensorFlow, with a specific check for GPUs. This ensures that the CUDA and cuDNN libraries are correctly installed and that the GPU is exposed to the framework.

2. **Functional benchmark**  
   We execute a large-scale matrix multiplication (`10000 x 10000`) both on the CPU and on the GPU. 

### Interpretation
- If a GPU device is listed and the GPU computation executes without error, the environment is correctly configured for hardware acceleration.  
- Comparing runtimes provides an empirical indication of the performance benefit when using the GPU over the CPU for large linear algebra operations, which are the core workload in deep learning models.  
- If no GPU is available, or if execution on `/GPU:0` fails, TensorFlow will default to CPU execution.

In [2]:

print("Dispositivos físicos:", tf.config.list_physical_devices())
print("GPU disponible:", tf.test.is_gpu_available(cuda_only=True))
print("Nombre GPU:", tf.test.gpu_device_name())


Dispositivos físicos: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
GPU disponible: True
Nombre GPU: /device:GPU:0


In [3]:
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    print("Dispositivo detectado:", gpu)

from tensorflow.python.client import device_lib
print("\nDetalle de dispositivos:")
print(device_lib.list_local_devices())


Dispositivo detectado: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Detalle de dispositivos:
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 1146023321909595114
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 5518655488
locality {
  bus_id: 1
  links {
  }
}
incarnation: 17975178256996543049
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 5060, pci bus id: 0000:01:00.0, compute capability: 12.0"
xla_global_id: 416903419
]


In [4]:
# CPU
with tf.device('/CPU:0'):
    a = tf.random.normal([10000, 10000])
    b = tf.random.normal([10000, 10000])
    start = time.time()
    c = tf.matmul(a, b)
    _ = c.numpy()  # fuerza el cálculo
    print("CPU:", time.time() - start, "segundos")

# GPU
with tf.device('/GPU:0'):
    a = tf.random.normal([10000, 10000])
    b = tf.random.normal([10000, 10000])
    start = time.time()
    c = tf.matmul(a, b)
    _ = c.numpy()
    print("GPU:", time.time() - start, "segundos")


CPU: 6.385891914367676 segundos
GPU: 2.1352758407592773 segundos
